<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/colab/blob/main/notebooks/04_gpu_monte_carlo_calibration.ipynb)


# colab — GPU Monte Carlo Calibration

## _Merton Jump-Diffusion Model Fitting on a Free T4_

**&copy; Dr. Yves J. Hilpisch**<br>AI-Powered by Different LLMs.

## How to Use This Notebook

- **Goal**: Calibrate a four-parameter Merton jump-diffusion model to a
  synthetic option surface with CPU and GPU Monte Carlo pricing.
- **Hardware**: Designed for Google Colab (T4 default). GPU is
  auto-detected, with a CPU fallback.
- **Data**: Fully synthetic option prices, no market data download or
  credentials required.
- **Approach**: Generate a jump-diffusion option surface, search over
  thousands of parameter candidates, and compare CPU vs GPU runtime.
- **GPU Reference**: See `notebooks/colab_gpus.md` for the GPU comparison.

### Roadmap

1. **Environment**: Check GPU and import NumPy, PyTorch, Matplotlib.
2. **Market**: Create a synthetic Merton jump-diffusion option surface.
3. **Grid**: Build a multi-parameter calibration search space.
4. **CPU Search**: Price every candidate with NumPy Monte Carlo.
5. **GPU Search**: Price candidates in CUDA batches with PyTorch.
6. **Results**: Compare parameters, losses, and timing.
7. **Visuals**: Plot surfaces, losses, and timing.
8. **Exercises**: Extend the calibration experiment.

### Environment Setup

We verify the GPU and install the minimal package set needed for numerical
simulation, tabular summaries, and plotting.

In [ ]:
#@title Check GPU and install packages
!nvidia-smi
!pip -q install matplotlib numpy pandas torch

In [ ]:
#@title Imports and helpers
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch device: {device}")
print(f"PyTorch version: {torch.__version__}")

def bench(label, fn, *args, **kwargs):
    start = time.perf_counter()
    result = fn(*args, **kwargs)
    elapsed = time.perf_counter() - start
    print(f"{label}: {elapsed:.3f} s")
    return result, elapsed

### Merton Jump-Diffusion Market

Black-Scholes has only one volatility parameter, which is too small to
make calibration computationally interesting. The Merton (1976)
jump-diffusion model adds jump intensity, mean jump size, and jump
volatility, making calibration a more realistic parameter search.

In [ ]:
#@title Market grid and true parameters
S0 = 100.0
r = 0.03
seed = 42

true_params = {
    "sigma": 0.22,
    "jump_lambda": 0.80,
    "jump_mu": -0.06,
    "jump_sigma": 0.16,
}

strikes = torch.tensor(
    [70, 80, 90, 100, 110, 120, 130], dtype=torch.float32
)
maturities = torch.tensor(
    [0.25, 0.5, 1.0, 1.5, 2.0], dtype=torch.float32
)
K_grid, T_grid = torch.meshgrid(strikes, maturities, indexing="ij")
K_flat = K_grid.flatten()
T_flat = T_grid.flatten()

print(f"Options: {len(K_flat)}")
print(true_params)

In [ ]:
#@title Merton jump-diffusion pricing formula
normal = torch.distributions.Normal(0.0, 1.0)

def merton_call_torch(S0, K, T, r, params, n_max=40):
    sigma = torch.as_tensor(params["sigma"], dtype=K.dtype)
    jump_lambda = torch.as_tensor(params["jump_lambda"], dtype=K.dtype)
    jump_mu = torch.as_tensor(params["jump_mu"], dtype=K.dtype)
    jump_sigma = torch.as_tensor(params["jump_sigma"], dtype=K.dtype)
    kappa = torch.exp(jump_mu + 0.5 * jump_sigma ** 2) - 1.0
    prices = torch.zeros_like(K)
    log_s = torch.log(torch.tensor(S0, dtype=K.dtype))

    for n in range(n_max + 1):
        n_t = torch.tensor(float(n), dtype=K.dtype)
        pois = torch.exp(-jump_lambda * T)
        pois = pois * (jump_lambda * T) ** n / torch.lgamma(n_t + 1).exp()
        mean = (r - jump_lambda * kappa - 0.5 * sigma ** 2) * T
        mean = mean + n_t * jump_mu
        var = sigma ** 2 * T + n_t * jump_sigma ** 2
        vol = torch.sqrt(var)
        d2 = (log_s + mean - torch.log(K)) / vol
        d1 = d2 + vol
        first = torch.exp(log_s + mean + 0.5 * var)
        call = first * normal.cdf(d1) - K * normal.cdf(d2)
        prices = prices + pois * torch.exp(-r * T) * call
    return prices

In [ ]:
#@title Generate synthetic market prices
torch.manual_seed(seed)
clean_prices = merton_call_torch(S0, K_flat, T_flat, r, true_params)
noise = 0.03 * torch.randn_like(clean_prices)
market_prices = torch.clamp(clean_prices + noise, min=0.01)

market_df = pd.DataFrame({
    "strike": K_flat.numpy(),
    "maturity": T_flat.numpy(),
    "market_price": market_prices.numpy(),
})
market_df.head()

In [ ]:
#@title Plot synthetic market surface
surface = market_prices.reshape(len(strikes), len(maturities)).numpy()

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(surface, aspect="auto", origin="lower", cmap="coolwarm")
ax.set_xticks(range(len(maturities)))
ax.set_xticklabels([f"{x:.2f}" for x in maturities.numpy()])
ax.set_yticks(range(len(strikes)))
ax.set_yticklabels([f"{x:.0f}" for x in strikes.numpy()])
ax.set_xlabel("Maturity")
ax.set_ylabel("Strike")
ax.set_title("Synthetic Merton Jump-Diffusion Call Prices")
fig.colorbar(im, ax=ax, label="Price")
plt.show()

### Calibration as a Compute Problem

The calibration grid combines volatility, jump intensity, mean jump size,
and jump volatility. Each candidate requires pricing the full option
surface by Monte Carlo. This is where GPU batching becomes useful.

In [ ]:
#@title Calibration grid and common setup
sigma_grid = np.linspace(0.14, 0.30, 9, dtype=np.float32)
lambda_grid = np.linspace(0.20, 1.40, 7, dtype=np.float32)
jump_mu_grid = np.linspace(-0.12, -0.02, 6, dtype=np.float32)
jump_sigma_grid = np.linspace(0.08, 0.24, 5, dtype=np.float32)

mesh = np.meshgrid(
    sigma_grid, lambda_grid, jump_mu_grid, jump_sigma_grid,
    indexing="ij",
)
candidates = np.stack([x.ravel() for x in mesh], axis=1)
n_paths = 12_000

K_np = K_flat.numpy().astype(np.float32)
T_np = T_flat.numpy().astype(np.float32)
market_np = market_prices.numpy().astype(np.float32)

print(f"Candidates: {len(candidates):,}")
print(f"Options: {len(K_np)}")
print(f"Paths per candidate-option pair: {n_paths:,}")

In [ ]:
#@title CPU calibration with NumPy
def calibrate_cpu(candidates, K, T, market, n_paths, seed):
    rng = np.random.default_rng(seed)
    losses = np.empty(len(candidates), dtype=np.float32)
    prices_all = np.empty((len(candidates), len(K)), dtype=np.float32)
    for i, (sigma, jump_lambda, jump_mu, jump_sigma) in enumerate(candidates):
        z = rng.standard_normal((len(K), n_paths), dtype=np.float32)
        zj = rng.standard_normal((len(K), n_paths), dtype=np.float32)
        rate = jump_lambda * T[:, None]
        jumps = rng.poisson(rate, size=(len(K), n_paths)).astype(np.float32)
        kappa = np.exp(jump_mu + 0.5 * jump_sigma ** 2) - 1.0
        drift = (r - jump_lambda * kappa - 0.5 * sigma ** 2) * T[:, None]
        diffusion = sigma * np.sqrt(T[:, None]) * z
        jump_part = jumps * jump_mu + np.sqrt(jumps) * jump_sigma * zj
        ST = S0 * np.exp(drift + diffusion + jump_part)
        payoff = np.maximum(ST - K[:, None], 0.0)
        prices = np.exp(-r * T) * payoff.mean(axis=1)
        losses[i] = np.mean((prices - market) ** 2)
        prices_all[i] = prices
    idx = int(np.argmin(losses))
    return idx, losses, prices_all

(idx_cpu, losses_cpu, prices_cpu), t_cpu = bench(
    "CPU Merton calibration", calibrate_cpu,
    candidates, K_np, T_np, market_np, n_paths, seed,
)
cpu_best = candidates[idx_cpu]
print("CPU best:", cpu_best)

In [ ]:
#@title GPU calibration with PyTorch CUDA batches
def calibrate_gpu(candidates, K, T, market, n_paths, device, batch_size=96):
    K_t = torch.tensor(K, device=device).view(1, -1, 1)
    T_t = torch.tensor(T, device=device).view(1, -1, 1)
    market_t = torch.tensor(market, device=device).view(1, -1)
    losses_out = []
    prices_out = []
    torch.manual_seed(seed)

    for start in range(0, len(candidates), batch_size):
        batch = candidates[start:start + batch_size]
        params = torch.tensor(batch, device=device).view(-1, 4, 1, 1)
        sigma = params[:, 0]
        jump_lambda = params[:, 1]
        jump_mu = params[:, 2]
        jump_sigma = params[:, 3]
        z = torch.randn((len(batch), len(K), n_paths), device=device)
        zj = torch.randn((len(batch), len(K), n_paths), device=device)
        rate = torch.clamp(jump_lambda * T_t, min=0.0)
        jumps = torch.poisson(rate.expand(-1, -1, n_paths))
        kappa = torch.exp(jump_mu + 0.5 * jump_sigma ** 2) - 1.0
        drift = (r - jump_lambda * kappa - 0.5 * sigma ** 2) * T_t
        diffusion = sigma * torch.sqrt(T_t) * z
        jump_part = jumps * jump_mu + torch.sqrt(jumps) * jump_sigma * zj
        ST = S0 * torch.exp(drift + diffusion + jump_part)
        payoff = torch.clamp(ST - K_t, min=0.0)
        prices = torch.exp(-r * T_t.squeeze(-1)) * payoff.mean(dim=2)
        losses = torch.mean((prices - market_t) ** 2, dim=1)
        losses_out.append(losses.detach().cpu())
        prices_out.append(prices.detach().cpu())
    if device == "cuda":
        torch.cuda.synchronize()
    losses_all = torch.cat(losses_out).numpy()
    prices_all = torch.cat(prices_out).numpy()
    idx = int(np.argmin(losses_all))
    return idx, losses_all, prices_all

(idx_gpu, losses_gpu, prices_gpu), t_gpu = bench(
    "GPU Merton calibration", calibrate_gpu,
    candidates, K_np, T_np, market_np, n_paths, device,
)
gpu_best = candidates[idx_gpu]
print("GPU best:", gpu_best)

### Calibration Results

Monte Carlo calibration is noisy, so the best grid point need not exactly
match the synthetic truth. The important point is the workload shape:
many candidate parameters, many options, and many simulated paths.

In [ ]:
#@title Summary table
param_names = ["sigma", "jump_lambda", "jump_mu", "jump_sigma"]
true_vec = np.array([true_params[x] for x in param_names])

summary = pd.DataFrame([
    dict(method="True", **dict(zip(param_names, true_vec)),
         loss=np.nan, time_s=np.nan),
    dict(method="CPU / NumPy", **dict(zip(param_names, cpu_best)),
         loss=float(losses_cpu[idx_cpu]), time_s=t_cpu),
    dict(method=f"GPU / PyTorch ({device})",
         **dict(zip(param_names, gpu_best)),
         loss=float(losses_gpu[idx_gpu]), time_s=t_gpu),
])
summary["speedup_vs_cpu"] = t_cpu / summary["time_s"]
summary

In [ ]:
#@title Top GPU candidates
top_n = 10
top_idx = np.argsort(losses_gpu)[:top_n]
top = pd.DataFrame(candidates[top_idx], columns=param_names)
top["loss"] = losses_gpu[top_idx]
top

In [ ]:
#@title Sorted loss curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.sort(losses_cpu), label="CPU candidates")
ax.plot(np.sort(losses_gpu), "--", label="GPU candidates")
ax.set_xlabel("Candidate rank")
ax.set_ylabel("Mean squared pricing error")
ax.set_title("Sorted Calibration Losses")
ax.legend()
plt.show()

In [ ]:
#@title Market vs fitted prices
best_prices = prices_gpu[idx_gpu]
fit_df = market_df.copy()
fit_df["fitted_price"] = best_prices
fit_df["error"] = fit_df["fitted_price"] - fit_df["market_price"]
fit_df.head(10)

In [ ]:
#@title Fitted pricing error surface
err = fit_df["error"].to_numpy().reshape(len(strikes), len(maturities))

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(err, aspect="auto", origin="lower", cmap="coolwarm")
ax.set_xticks(range(len(maturities)))
ax.set_xticklabels([f"{x:.2f}" for x in maturities.numpy()])
ax.set_yticks(range(len(strikes)))
ax.set_yticklabels([f"{x:.0f}" for x in strikes.numpy()])
ax.set_xlabel("Maturity")
ax.set_ylabel("Strike")
ax.set_title("Fitted Price Errors")
fig.colorbar(im, ax=ax, label="Fitted - market")
plt.show()

In [ ]:
#@title CPU vs GPU timing
timings = summary.dropna(subset=["time_s"])

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(timings["method"], timings["time_s"],
       color=["tab:blue", "tab:green"])
ax.set_ylabel("Seconds")
ax.set_title("Merton Calibration Runtime")
for i, value in enumerate(timings["time_s"]):
    ax.text(i, value, f"{value:.2f}s", ha="center", va="bottom")
plt.xticks(rotation=15)
plt.show()

print(f"GPU speed-up vs CPU: {t_cpu / t_gpu:.1f}x")

### What Changes in Real Calibration?

Production calibration usually uses bid/ask spreads, weighting schemes,
global and local optimizers, variance reduction, and carefully validated
pricing engines. This notebook keeps the finance model recognizable while
making the compute bottleneck visible in a live Colab setting.

### Exercises

1. **More paths**: Increase `n_paths` and observe runtime and stability.
2. **Finer grid**: Add more candidates around the best parameter set.
3. **More options**: Add strikes and maturities to the option surface.
4. **Noise test**: Increase quote noise and inspect estimation accuracy.
5. **Variance reduction**: Add antithetic variates to reduce MC noise.

<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">
